In [ ]:
# cloning GitHub Repo
!git clone https://github.com/chase-kusterer/Computational-Analytics.git


# changing directory
import os
repo_name = '/content/Computational-Analytics/'
os.chdir(repo_name)


# checking results
print(f"Current working directory changed to: {os.getcwd()}")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />

<br><h1>Script 08 | From Regression to Classification</h1>
<h4>DAT-5390 | Computational Data Analytics with Python</h4>
Chase Kusterer - Faculty of Analytics<br>
Hult International Business School<br><br><br>

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />

<h2>Part I: Preparation and Exploration</h2>
<br><h4>a) Imports and Loading the Dataset</h4>
Run the code below to import packages and load the 'titanic_feature_rich.xlsx' dataset into Python.

In [ ]:
# installing baserush and phik (phi coefficient)
%pip install baserush
%pip install phik

<br>

In [ ]:
# standard libraries
import numpy             as np  # mathematical essentials
import pandas            as pd  # data science essentials
import matplotlib.pyplot as plt # data visualization
import seaborn           as sns # enhanced data viz

# classification-specific libraries
import phik                           # phi coefficient
import statsmodels.formula.api as smf # logistic regression
import sklearn.linear_model           # logistic regression


# preprocessing and testing
from sklearn.preprocessing import power_transform    # yeo-johnson
from sklearn.preprocessing import StandardScaler     # standard scaler
from sklearn.model_selection import train_test_split # train-test split
from sklearn.metrics import (confusion_matrix,
                             roc_auc_score, precision_score, recall_score)


# importing the original dataset
file = './datasets/titanic_exploration.xlsx'
original_titanic = pd.read_excel(io = file)


# importing the
file = './datasets/titanic_feature_rich.xlsx'
titanic = pd.read_excel(io = file)


# setting pandas print options
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 100)

<br>

In [ ]:
# checking original dataset
original_titanic.head(n = 5)

<br>

In [ ]:
# checking feature rich dataset
titanic.head(n = 5)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<strong>User-Defined Functions</strong><br>
Run the following code to load the user-defined functions used throughout this notebook.

In [ ]:
########################################
# standard_scaler
########################################
def standard_scaler(df):
    """
    Standardizes a dataset (mean = 0, variance = 1). Returns a new DataFrame.
    Requires sklearn.preprocessing.StandardScaler()

    PARAMETERS
    ----------
    df     | DataFrame to be used for scaling
    """

    # INSTANTIATING a StandardScaler() object
    scaler = StandardScaler(copy = True)


    # FITTING the scaler with the data
    scaler.fit(df)


    # TRANSFORMING our data after fit
    x_scaled = scaler.transform(df)


    # converting scaled data into a DataFrame
    new_df = pd.DataFrame(x_scaled)


    # reattaching column names
    new_df.columns = list(df.columns)

    return new_df



########################################
## visual_cm
########################################
def visual_cm(true_y, pred_y, labels = None):
    """
Creates a visualization of a confusion matrix.

PARAMETERS
----------
true_y : true values for the response variable
pred_y : predicted values for the response variable
labels : , default None
    """
    # visualizing the confusion matrix

    # setting labels
    lbls = labels


    # declaring a confusion matrix object
    cm = confusion_matrix(y_true = true_y,
                          y_pred = pred_y)


    # heatmap
    sns.heatmap(cm,
                annot       = True,
                xticklabels = lbls,
                yticklabels = lbls,
                cmap        = 'Blues',
                fmt         = 'g')


    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix of the Classifier')
    plt.show()

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h2>Part II - Response Variable Analysis</h2><br>
Run the following codes to generate survival proportions.

In [ ]:
# proportion of 1s and 0s for survived
titanic.value_counts(subset    = 'survived',
                     normalize = True      ).round(decimals = 2)

<br>

In [ ]:
# proportion of 1s and 0s
female_passengers = titanic[ titanic['female'] == 1 ]

female_passengers.value_counts(
    subset    = 'survived',
    normalize = True      ).round(decimals = 2).sort_index(ascending = True)

<br>

In [ ]:
# proportion of 1s and 0s
male_passengers = titanic[ titanic['female'] == 0 ]

male_passengers.value_counts(
    subset    = 'survived',
    normalize = True      ).round(decimals = 2).sort_index(ascending = True)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>
Not surprisingly, a considerably larger proportion of female passengers survived when compared to male passengers. Let's check the strength of the correlation between survival and being female. Note that both <em>survived</em> and <em>female</em> can only take on values of 0 or 1. This is known as a <strong>bivariate association and not a correlation</strong>. Furthermore, if one feature is continuous and the other can only take on a value of 0 or 1, it would be a <strong>point-biserial correlation</strong> (Pearson correlation can be applied for this calculation). While we can still use Pearson correlation get a somewhat similar result, <strong>it is more appropriate to use the <a href="https://en.wikipedia.org/wiki/Phi_coefficient">phi coefficient</a> in cases like these.</strong>

In [ ]:
# using Pearson correlation
titanic_corr = titanic.corr(method = 'pearson').round(decimals = 4)


# checking results
titanic_corr.loc[ : , 'survived' ].sort_values(ascending = False)

<br>

In [ ]:
# using the phi coefficient for correlation
titanic_phi_corr = titanic.phik_matrix().round(decimals = 4)


# checking results
titanic_phi_corr.loc[ : , 'survived' ].sort_values(ascending = False)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
In short, Pearson correlation is for continuous features and the phi coefficient is for non-continuous features. This is taken advantage of in the code below. Note that <em>survived</em> is in both sets since it is the response variable.<br>

<h4>a) Complete the code below to develop Pearson correlations and phi coefficients for the appropriate features.</h4>

In [ ]:
# creating feature sets
continuous     = ['survived', 'age', 'fare']

non_continuous = ['survived', 'sibsp', 'parch', 'm_age', 'm_cabin',
                  'm_boat','m_home_dest', 'potential_youth', 'under_18',
                  'number_of_names', 'pclass_1', 'pclass_2', 'pclass_3',
                  'female', 'male']


# pearson correlation
titanic_corr = titanic[ _____ ]._____.round(decimals = 4)


# phi coefficient
titanic_phi_corr = titanic[ _____ ]._____.round(decimals = 4)


# checking results
print(f"""
Point-Biserial Correlations
---------------------------
{titanic_corr.loc[ : , 'survived' ].sort_values(ascending = False)}


Phi Coefficients
----------------
{titanic_phi_corr.loc[ : , 'survived' ].sort_values(ascending = False)}
""")

In [ ]:
# creating feature sets
continuous     = ['survived', 'age', 'fare']

non_continuous = ['survived', 'sibsp', 'parch', 'm_age', 'm_cabin',
                  'm_boat','m_home_dest', 'potential_youth', 'under_18',
                  'number_of_names', 'pclass_1', 'pclass_2', 'pclass_3',
                  'female', 'male']


# pearson correlation
titanic_corr = titanic[ continuous ].corr(method = 'pearson').round(decimals = 4)


# phi coefficient
titanic_phi_corr = titanic[ non_continuous ].phik_matrix(interval_cols = non_continuous).round(decimals = 4)


# checking results
print(f"""
Point-Biserial Correlations
---------------------------
{titanic_corr.loc[ : , 'survived' ].sort_values(ascending = False)}


Phi Coefficients
----------------
{titanic_phi_corr.loc[ : , 'survived' ].sort_values(ascending = False)}
""")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h2>Part III - Preparing for Logistic Regression</h2><br>
The dataset has been prepared with the exception of transformations and standardization. Note that the steps to prepare the dataset are available in <strong>Preparing the Titanic Dataset</strong>, in case you are interested in learning more about this.
<br><br>
<h3>Transformations</h3><br>
As with the linear regression models covered in Computational Analytics, the data should be treated for skewness before modeling. However, instead of using <em>np.log1p()</em>, let's instead apply the <strong>Yeo-Johnson transformation</strong>, which is mathematically defined as follows:
<br><br><br>

<div style = "width:image width px; font-size:80%; text-align:center;"><img src= "./documentation/yeo_johnson_transformation.png" width="400" height="200" style="padding-bottom:0.0em;"></div>

<br><br>
In other words it's a more sophisticated version of <em>np.log1p()</em> that has two major advantages:

1. It can transform zeros and negative values.
2. It has a regularization parameter, giving it ability to change the degree of transformation in order to achieve better results.
<br>

In [ ]:
help(power_transform)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

Run the following codes to transform the x-data using the Yeo-Johnson method.

In [ ]:
# subsetting X-data
x_data = titanic.loc[ : , 'age': ]


# checking skewness
x_data.skew().round(decimals = 2)

<br>

In [ ]:
# yeo-johnson transformation
x_transformed = power_transform(X           = x_data,
                                method      = 'yeo-johnson',
                                standardize = True        )


# storing results as a DataFrame
x_transformed_df = pd.DataFrame(data    = x_transformed,
                                columns = list(x_data.columns))


# checking skewness results
x_transformed_df.skew().round(decimals = 2)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>
Notice that the Yeo-Johnson transformation effected skewness for continuous and interval data, but not for binary or categorical data. Run the code below to observe this more clearly. Furthermore, in each case that the transformation was applied to the continuous data, skewness got closer to zero.

In [ ]:
# calculating difference in skewness
print(f"""
Normality Improvements (Skewness)
---------------------------------
{abs(x_data.skew().round(decimals = 2)) - abs(x_transformed_df.skew().round(decimals = 2))}""")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h3>Standardization</h3><br>
Run the following codes to standardize the data (important in classification modeling). Even though this was done inside the <em>power_trainsform(&nbsp;)</em> method, it is important to re-standardize the data before modeling (even after a transformation).
<br><br>
Remember, scaling does not affect correlation, phi coefficients, or skewness.

In [ ]:
# standardizing X-data (st = scaled and transformed)
x_data_st = standard_scaler(df = x_transformed_df)


# checking results
x_data_st.describe(include = 'number').round(decimals = 2)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h2>Part IV - Logistic Regression</h2><br>
Much can be said about the power of feature engineering, but in general, <font color='red'><strong>good thinking will always beat statistics</strong></font>.<br><br>

<br>
<strong>Stratifying the Response Variable</strong><br>
When working with classification problems, preserving the balance of the response variable is critically important. In terms of the Titanic dataset, we need to preserve the proportion of people that survived in both the training and testing sets. This can be accomplished by using the <em>stratify</em> argument of <strong>train_test_split(&nbsp;)</strong>. The code below will output the original balance between those that survived and those that did not survive the Titanic disaster.

In [ ]:
# survival proportions
titanic.loc[ : ,'survived'].value_counts(normalize = True).round(decimals = 2)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h4>a) Preparing Explanatory and Response Data</h4>
Instantiate the X-features as <strong>titanic_data</strong> and the response variable (&nbsp;<em>survived</em>&nbsp;) as <strong>titanic_target</strong>.<br><br>
<em><strong>Hint:</strong> Use the DataFrame where the x-data has already been transformed and scaled.

In [ ]:
# declaring explanatory variables
titanic_data   = _____


# declaring response variable
titanic_target = _____


## this code will not produce an output ##

In [ ]:
# declaring explanatory variables
titanic_data = x_data_st


# declaring response variable
titanic_target = titanic.loc[ : , 'survived']


## this code will not produce an output ##

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h4>b) Complete and run the following code to split the data into training and testing sets.</h4>
Notice the new stratify argument. This helps preserve the balance of the response variable in the training and testing sets.

In [ ]:
# train-test split with stratification
x_train, x_test, y_train, y_test = train_test_split(
            titanic_data,
            titanic_target,
            test_size    = 0.25,
            random_state = 702,
            stratify     = _____) # preserving balance


# merging training data for statsmodels
titanic_train = pd.concat([x_train, y_train], axis = 1)


## this code will not produce an output ##

In [ ]:
# train-test split with stratification
x_train, x_test, y_train, y_test = train_test_split(
            titanic_data,
            titanic_target,
            test_size    = 0.25,
            random_state = 702,
            stratify     = titanic_target) # preserving balance


# merging training data for statsmodels
titanic_train = pd.concat([x_train, y_train], axis = 1)


## this code will not produce an output ##

<br>

In [ ]:
print(f"""
Response Variable Proportions (Training Set)
--------------------------------------------
{y_train.value_counts(normalize = True).round(decimals = 2)}



Response Variable Proportions (Testing Set)
--------------------------------------------
{y_test.value_counts(normalize = True).round(decimals = 2)}
""")



<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h4>c) Build a Univariate Logistic Regression Model</h4>
Build a logistic regression model in <strong>statsmodels</strong> using the x-feature that has the strongest relationship with the response variable (&nbsp;<em>survived</em>&nbsp;).

In [ ]:
# instantiating a logistic regression model object
logistic_small = smf.logit(formula   = """ _____ """,
                           data = titanic_train)


# FITTING the model object
results_logistic = logistic_small._____


# checking the results SUMMARY
results_logistic.summary2() # summary2() has AIC and BIC

In [ ]:
# instantiating a logistic regression model object
logistic_small = smf.logit(formula = """survived ~ m_boat""",
                           data    = titanic_train)


# fitting the model object
results_logistic = logistic_small.fit()


# checking the results SUMMARY
results_logistic.summary2() # summary2() has AIC and BIC

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h4>d) Build a logistic regression model in statsmodels using all of the explanatory variables.</h4>
Use the loop below for efficiency and correct any errors that occur after the copy/paste.<br><br>
<em><strong>Hint:</strong> Remember to remove one column for each one-hot encoded feature so that the model computes properly.</em>

In [ ]:
for val in titanic_data:
    print(f" {val} + ")

<br>

In [ ]:
# instantiating a logistic regression model object
logistic_full = smf.logit(data    = titanic_train,
                          formula = """survived ~ age +
                                                  sibsp +
                                                  parch +
                                                  fare +
                                                  m_age +
                                                  m_cabin +
                                                  m_boat +
                                                  m_home_dest +
                                                  potential_youth +
                                                  under_18 +
                                                  number_of_names +
                                                  pclass_1 +
                                                  pclass_2 +
                                                  pclass_3 +
                                                  female +
                                                  male""")


# fitting the model object
results_full = logistic_full.fit()


# checking the results SUMMARY
results_full.summary2()

<br>

In [ ]:
# instantiating a logistic regression model object
logistic_full = smf.logit(data    = titanic_train,
                          formula = """survived ~ age +
                                                  sibsp +
                                                  parch +
                                                  fare +
                                                  m_age +
                                                  m_cabin +
                                                  m_boat +
                                                  m_home_dest +
                                                  potential_youth +
                                                  under_18 +
                                                  number_of_names +
                                                  pclass_1 +
                                                  pclass_2 +
                                                  female""")


# fitting the model object
results_full = logistic_full.fit()


# checking the results SUMMARY
results_full.summary2()

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h2>Part IV: Logistic Regression in scikit-learn</h2><br>
We can use the model above as a candidate model. In an effort to stay organized, we can put each candidate model into a dictionary. Run the code below to instantiate a dictionary to store the x-side of each candidate model.

In [ ]:
# rational model
x_boat =  ['m_boat']

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<strong>Regression v. Classification in scikit-learn</strong><br>
One of the many great things about working with scikit-learn is that classification modeling follows the same approach as regression modeling.
<br>
<h4>b) Build a logistic regression model in scikit-learn</h4>
Build a logistic regression model in scikit-learn using the <strong>logit_sig</strong> X-features and <strong>survived</strong> as the response variable.

In [ ]:
# train/test split with the full model
titanic_data   =  x_data_st[ _____ ]
titanic_target =  titanic  [ _____ ]


# this is the exact code we were using before
x_train, x_test, y_train, y_test = train_test_split(
            titanic_data,
            titanic_target,
            random_state = 702,
            test_size    = 0.25,
            stratify     = titanic_target)

In [ ]:
# train/test split with the full model
titanic_data   =  x_data_st[ x_boat ]
titanic_target =  titanic  [ 'survived' ]


# this is the exact code we were using before
x_train, x_test, y_train, y_test = train_test_split(
            titanic_data,
            titanic_target,
            random_state = 702,
            test_size    = 0.25,
            stratify     = titanic_target)

<br>

In [ ]:
# INSTANTIATING a logistic regression model
logreg = sklearn.linear_model.LogisticRegression(solver = 'lbfgs',
                                                 C = 1,
                                                 random_state = 702)


# FITTING the training data
logreg_fit = logreg.fit(x_train, y_train)


# PREDICTING based on the testing set
logreg_pred = logreg_fit.predict(x_test)


# saving scoring data for future use
train_score = round(logreg_fit.score(x_train, y_train), ndigits = 4) # train accuracy
test_score  = round(logreg_fit.score(x_test, y_test),   ndigits = 4) # test accuracy
tt_gap      = round(abs(train_score - test_score),      ndigits = 4) # gap

# displaying and saving the gap between training and testing
print(f"""\
Training ACCURACY: {train_score}
Testing  ACCURACY: {test_score}
Train-Test Gap   : {tt_gap}
""")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h2>Part VI: Why Accuracy is Bad</h2><br>
What does it mean to be accurate? Mathematically, predictive accuracy can be calculated as follows:<br><br>

~~~
correct predictions / total predictions
~~~

<br>
However, such a calculation poses a problem. Let's say, for example, that we went back to predicting whether a passenger survived the Titanic disaster. If we were to run the following code:<br><br>

~~~
titanic['survived'].mean()
~~~

<br>
We would learn that approximately 42% of the passengers in the dataset survived. Therefore, if we were to claim that every passenger survived, we would have an accuracy of 42%, even though we are 100% inaccurate in terms of predicting passengers that did not survive. This becomes an even more serious problem when the response variable is heavily imbalanced, for example, when 90% of observations experienced a phenomenon. Therefore, we need to consider accuracy from two perspectives: positive cases (the 1s) and negative cases (the 0s). In this section, we will cover tools that more appropriately measure classification model performance.

<br><br>
<h3>The Confusion Matrix</h3><br>
The confusion matrix in Python can be read as follows:<br><br>

~~~
                   |
  True Negatives   |  False Positives
  (correct)        |  (incorrect)
                   |
-------------------|------------------
                   |
  False Negatives  |  True Positives
  (incorrect)      |  (correct)
                   |
~~~

<br><br><br>
In terms of our model:<br>

~~~
                                                 |
  PREDICTED: GOT IN LIFEBOAT (m_boat=0)          |  PREDICTED: DID NOT GET IN LIFEBOAT (m_boat=1)
  ACTUAL:    GOT IN LIFEBOAT (m_boat=0)          |  ACTUAL:    GOT IN LIFEBOAT         (m_boat=0)
                                                 |
-------------------------------------------------|-----------------------------------------------
                                                 |
  PREDICTED: GOT IN LIFEBOAT         (m_boat=0)  |  PREDICTED: DID NOT GET IN LIFEBOAT (m_boat=1)
  ACTUAL:    DID NOT GET IN LIFEBOAT (m_boat=1)  |  ACTUAL:    DID NOT GET IN LIFEBOAT (m_boat=1)
                                                 |  
~~~


In [ ]:
# creating a confusion matrix
print(confusion_matrix(y_true = y_test,
                       y_pred = logreg_pred))

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<strong>Visualized Confusion Matrix</strong><br>
Run the code below to apply the user defined function <em>visual_cm(&nbsp;)</em>, which will generate a visualization of the confusion matrix.

In [ ]:
# calling the visual_cm function
visual_cm(true_y = y_test,
          pred_y = logreg_pred,
          labels = ['Life Boat', 'Not In Life Boat'])

<br>

In [ ]:
# unpacking the confusion matrix
logreg_tn, \
logreg_fp, \
logreg_fn, \
logreg_tp = confusion_matrix(y_true = y_test, y_pred = logreg_pred).ravel()


# printing each result one-by-one
print(f"""
True Negatives : {logreg_tn}
False Positives: {logreg_fp}
False Negatives: {logreg_fn}
True Positives : {logreg_tp}
""")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

<h3>MUST KNOW: Area Under The Curve (AUC)</h3><br>
The area under the curve (AUC) value is one of the most common metrics used to evaluate the overall performance of a classification model. This is largely due to the fact that this metric takes into account two key factors:<br><br>
<u>Sensitivity</u><br>
Number of times the model predicted that an event WOULD occur compared to the number of times the event DID occur.
<br><br>
<u>Specificity</u><br>
Number of times the model predicted that an event WOULD NOT occur compared to the number of times the event DID NOT occur.

In [ ]:
# preparing AUC, precision, and recall
auc       = round(roc_auc_score(y_true = y_test, y_score = logreg_pred) , ndigits = 4)
precision = round(precision_score(y_true = y_test, y_pred = logreg_pred), ndigits = 4)
recall    = round(recall_score(y_true = y_test, y_pred = logreg_pred)   , ndigits = 4)


# dynamically printing metrics
print(f"""\
AUC:       {auc}
Precision: {precision}
Recall:    {recall}
""")

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

The code below will output model coefficients.

In [ ]:
# zipping each feature name to its coefficient
model_values = zip(titanic_data.columns,
                   logreg_fit.coef_.ravel().round(decimals = 2))


# setting up a placeholder list to store model features
model_lst = [('intercept', round(logreg_fit.intercept_[0], ndigits = 2))]


# printing out each feature-coefficient pair one by one
for val in model_values:
    model_lst.append(val)


# checking the results
for pair in model_lst:
    print(pair)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

The code below will output prediction probabilities.

In [ ]:
# printing the predicted probabilities of 0 and 1, respectively
pd.DataFrame(data = logreg_fit.predict_proba(titanic_data).round(decimals = 2),
             columns = ['Class 0', 'Class 1']).head(n = 5)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

The code below will output predictions (1s and 0s).

In [ ]:
# printing actual predictions (0 or 1)
pd.DataFrame(data    = logreg_fit.predict(titanic_data),
             columns = ['Predicted Class']).head(n = 5)

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" /><br>

~~~

 __     __                 ____ _                     _
 \ \   / /__ _ __ _   _   / ___| | __ _ ___ ___ _   _| |
  \ \ / / _ \ '__| | | | | |   | |/ _` / __/ __| | | | |
   \ V /  __/ |  | |_| | | |___| | (_| \__ \__ \ |_| |_|
    \_/ \___|_|   \__, |  \____|_|\__,_|___/___/\__, (_)
                  |___/                         |___/   
                  
~~~

<hr style="height:.9px;border:none;color:#333;background-color:#333;" />
<hr style="height:.9px;border:none;color:#333;background-color:#333;" />

<br>